In [ ]:
!pip install tqdm

In [ ]:
import os
import shutil
import hashlib
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
files.upload()

Saving DATASET FER 2013.zip to DATASET FER 2013.zip


In [ ]:
import zipfile

zip_path = "/content/DATASET FER 2013.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Selesai extract")

Selesai extract


In [ ]:
def hash_image(path):
    with Image.open(path) as img:
        img = img.convert("L").resize((48,48))
        return hashlib.md5(img.tobytes()).hexdigest()


def find_duplicates(folder):
    hash_map = defaultdict(list)

    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)

        if not os.path.isdir(cls_path):
            continue

        for file in os.listdir(cls_path):
            path = os.path.join(cls_path, file)

            h = hash_image(path)
            hash_map[h].append(path)

    duplicates = {
        h: paths for h, paths in hash_map.items()
        if len(paths) > 1
    }

    return duplicates

In [ ]:
print("Train path:", train_dir)
print("Val path:", val_dir)

print("Isi DATA_DIR:", os.listdir(extract_path))

Train path: /content/dataset/train
Val path: /content/dataset/validation
Isi DATA_DIR: ['images']


In [ ]:
train_dir = os.path.join(extract_path, "images", "train")
val_dir   = os.path.join(extract_path, "images", "validation")

train_dups = find_duplicates(train_dir)
val_dups   = find_duplicates(val_dir)

print("Duplikat di train:", len(train_dups))
print("Duplikat di validation:", len(val_dups))

Duplikat di train: 1055
Duplikat di validation: 77


In [ ]:
def get_hash_set(folder):
    hashes = set()

    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)

        if not os.path.isdir(cls_path):
            continue

        for file in os.listdir(cls_path):
            path = os.path.join(cls_path, file)
            hashes.add(hash_image(path))

    return hashes


train_hashes = get_hash_set(train_dir)
val_hashes   = get_hash_set(val_dir)

overlap = train_hashes & val_hashes

print("Jumlah overlap train-validation:", len(overlap))

Jumlah overlap train-validation: 510


In [ ]:
clean_val_dir = "/content/clean_validation"

os.makedirs(clean_val_dir, exist_ok=True)

for cls in os.listdir(val_dir):
    os.makedirs(os.path.join(clean_val_dir, cls), exist_ok=True)

for cls in os.listdir(val_dir):
    cls_path = os.path.join(val_dir, cls)

    for file in os.listdir(cls_path):
        path = os.path.join(cls_path, file)

        h = hash_image(path)

        if h not in train_hashes:
            shutil.copy(path, os.path.join(clean_val_dir, cls, file))

In [ ]:
shutil.make_archive("/content/clean_dataset", 'zip', extract_path)

'/content/clean_dataset.zip'

In [ ]:
from google.colab import files
files.download("/content/clean_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>